                                                                    ***TUGAS MANDIRI***

***PERSIAPAN SPRAKSESSION***

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col, lit
from datetime import datetime, timedelta
import os, shutil, time, json
import pandas as pd


spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara-Streaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


***A. MENYIAPKAN STREAMING SOURCE***

In [31]:
skema_pesanan = StructType([
    StructField("pesanan_id",   StringType(),    True),
    StructField("nama_cabang",  StringType(),    True),
    StructField("nama_menu",    StringType(),    True),
    StructField("qty",          IntegerType(),   True),
    StructField("harga_satuan", IntegerType(),   True),
    StructField("waktu_pesan",  TimestampType(), True),
])

folder_stream = "tugas10_stream"
folder_tmp    = "tugas10_tmp"     # tempat menulis berkas sebelum dipindah ke folder stream

# Bersihkan sisa eksekusi sebelumnya
for f in [folder_stream, folder_tmp, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)
os.makedirs(folder_tmp, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("df_stream.isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

df_stream.isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



***B. AGREGASI PER CABANG***

In [32]:
df_stream = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang_df = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang aktif:", query_cabang.isActive)

Query agregasi_cabang aktif: True


***C. AGREGASI BERBASIS WINDOW WAKTU***

In [35]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu_df = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agregasi_waktu_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu aktif:", query_waktu.isActive)

Query agregasi_waktu aktif: True


***D. MENGUJI DENGAN DATA SIMULASI***

In [36]:
semua_data = []

def verifikasi(no):
    query_cabang.processAllAvailable()
    query_waktu.processAllAvailable()

    hasil = spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang")
    print(f"Hasil Batch {no}:")
    hasil.show()

    pdf = pd.concat(semua_data)
    pdf["total_harga"] = pdf.qty * pdf.harga_satuan

    harapan = pdf.groupby("nama_cabang").agg(
        total_pendapatan=("total_harga", "sum"),
        jumlah_pesanan=("pesanan_id", "count")
    ).reset_index().sort_values("nama_cabang")

    aktual = hasil.toPandas().sort_values("nama_cabang")
    print("Perhitungan Pandas:")
    print(harapan.to_string(index=False))
    print("Sesuai? ->", harapan.equals(aktual), "\n")

In [41]:
b1 = pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003", "P004"],
    "nama_cabang": ["Magelang", "Solo", "Magelang", "Yogyakarta"],
    "nama_menu": ["Kopi Susu", "Americano", "Latte", "Kopi Susu"],
    "qty": [2, 1, 3, 4],
    "harga_satuan": [18000, 20000, 25000, 18000],
    "waktu_pesan": [fmt(t0 + timedelta(seconds=10))] * 4
})
b1.to_csv(f"{folder_stream}/batch_1.csv", index=False)
semua_data.append(b1)
print("Batch1 dikirim. Menunggu Spark memproses..")
time.sleep(5)
verifikasi(1)

Batch1 dikirim. Menunggu Spark memproses..


Hasil Batch 1:
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          111000|             2|
|       Solo|           20000|             1|
| Yogyakarta|           72000|             1|
+-----------+----------------+--------------+

Perhitungan Pandas:
nama_cabang  total_pendapatan  jumlah_pesanan
   Magelang            111000               2
       Solo             20000               1
 Yogyakarta             72000               1
Sesuai? -> True 



In [42]:
b2 = pd.DataFrame({
    "pesanan_id": ["P005", "P006", "P007"],
    "nama_cabang": ["Semarang", "Solo", "Magelang"],
    "nama_menu": ["Croissant", "Latte", "Americano"],
    "qty": [5, 2, 1],
    "harga_satuan": [22000, 25000, 20000],
    "waktu_pesan": [fmt(t0 + timedelta(seconds=70))] * 3
})
b2.to_csv(f"{folder_stream}/batch_2.csv", index=False)
semua_data.append(b2)
print("Batch2 dikirim. Menunggu Spark memproses..")
time.sleep(5)
verifikasi(2)

Batch2 dikirim. Menunggu Spark memproses..


Hasil Batch 2:
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          131000|             3|
|   Semarang|          110000|             1|
|       Solo|           70000|             2|
| Yogyakarta|           72000|             1|
+-----------+----------------+--------------+

Perhitungan Pandas:
nama_cabang  total_pendapatan  jumlah_pesanan
   Magelang            131000               3
   Semarang            110000               1
       Solo             70000               2
 Yogyakarta             72000               1
Sesuai? -> True 



In [43]:
b3 = pd.DataFrame({
    "pesanan_id": ["P008", "P009", "P010", "P011"],
    "nama_cabang": ["Yogyakarta", "Semarang", "Solo", "Yogyakarta"],
    "nama_menu": ["Americano", "Kopi Susu", "Croissant", "Latte"],
    "qty": [3, 2, 2, 1],
    "harga_satuan": [20000, 18000, 22000, 25000],
    "waktu_pesan": [fmt(t0 + timedelta(seconds=130))] * 4
})
b3.to_csv(f"{folder_stream}/batch_3.csv", index=False)
semua_data.append(b3)
print("Batch3 dikirim. Menunggu Spark memproses..")
time.sleep(5)
verifikasi(3)

Batch3 dikirim. Menunggu Spark memproses..


Hasil Batch 3:
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          131000|             3|
|   Semarang|          146000|             2|
|       Solo|          114000|             3|
| Yogyakarta|          157000|             3|
+-----------+----------------+--------------+

Perhitungan Pandas:
nama_cabang  total_pendapatan  jumlah_pesanan
   Magelang            131000               3
   Semarang            146000               2
       Solo            114000               3
 Yogyakarta            157000               3
Sesuai? -> True 



***E. SINK BERBASIS BERKAS (TANTANGAN)***

In [44]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("query3 (parquet sink) aktif:", query3.isActive)

query3 (parquet sink) aktif: True


In [47]:
b4 = pd.DataFrame({
    "pesanan_id":  ["P012", "P013"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu":   ["Kopi Susu", "Latte"],
    "qty":         [3, 2],
    "harga_satuan":[18000, 25000],
    "waktu_pesan": [fmt(t0 + timedelta(seconds=190))] * 2,
})
b4.to_csv(f"{folder_stream}/batch_4.csv", index=False)
semua_data.append(b4)
print("Batch 4 berhasil dikirim")
time.sleep(6)
query3.processAllAvailable()

Batch 4 berhasil dikirim


In [48]:
# Bukti 1: berkas Parquet benar-benar tersimpan
!ls tugas10_output

part-00000-201ea5b5-5d33-46b3-9589-7fa7cefd182e-c000.snappy.parquet
part-00000-4d9e91a2-ff59-4352-a3b5-62f9d8a6278f-c000.snappy.parquet
part-00001-47a61749-d9bd-44ca-8a0a-9876ab591b8b-c000.snappy.parquet
part-00002-c8bcb7d9-aefc-4247-9d68-84335cce3328-c000.snappy.parquet
_spark_metadata


In [49]:
# Bukti 2: dibaca kembali sebagai DataFrame BATCH biasa
df_hasil = spark.read.parquet("tugas10_output")
print("Apakah streaming?", df_hasil.isStreaming)
print("Jumlah baris di Parquet:", df_hasil.count(), "| jumlah baris yang dikirim:", sum(len(d) for d in semua_data))
df_hasil.orderBy("pesanan_id").show(20, truncate=False)

Apakah streaming? False


Jumlah baris di Parquet: 13 | jumlah baris yang dikirim: 17
+----------+-----------+---------+---+------------+-------------------+-----------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|waktu_pesan        |total_harga|
+----------+-----------+---------+---+------------+-------------------+-----------+
|P001      |Magelang   |Kopi Susu|2  |18000       |2026-11-08 21:35:15|36000      |
|P002      |Solo       |Americano|1  |20000       |2026-11-08 21:35:15|20000      |
|P003      |Magelang   |Latte    |3  |25000       |2026-11-08 21:35:15|75000      |
|P004      |Yogyakarta |Kopi Susu|4  |18000       |2026-11-08 21:35:15|72000      |
|P005      |Semarang   |Croissant|5  |22000       |2026-11-08 21:36:15|110000     |
|P006      |Solo       |Latte    |2  |25000       |2026-11-08 21:36:15|50000      |
|P007      |Magelang   |Americano|1  |20000       |2026-11-08 21:36:15|20000      |
|P008      |Yogyakarta |Americano|3  |20000       |2026-11-08 21:37:15|60000      |
|P009      |Sema

                                                ***EKSPLORASI KODE***

In [50]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

# Mengambil hasil agregasi cabang sebagai DataFrame batch
df_ranking = spark.table("agregasi_cabang")

# Membuat urutan ranking berdasarkan pendapatan
window_ranking = Window.orderBy(
    col("total_pendapatan").desc()
)

# Memberikan peringkat pada setiap cabang
hasil_ranking = df_ranking.withColumn(
    "peringkat",
    dense_rank().over(window_ranking)
)

# Menampilkan hasil ranking
hasil_ranking.orderBy("peringkat").show()

[Stage 40:>                                                         (0 + 1) / 1]

+-----------+----------------+--------------+---------+
|nama_cabang|total_pendapatan|jumlah_pesanan|peringkat|
+-----------+----------------+--------------+---------+
|   Magelang|          185000|             4|        1|
|       Solo|          164000|             4|        2|
| Yogyakarta|          157000|             3|        3|
|   Semarang|          146000|             2|        4|
+-----------+----------------+--------------+---------+



In [51]:
# Menghitung total produk terjual per cabang
eksplorasi_3 = df_stream.groupBy(
    "nama_cabang"
).agg(
    spark_sum("qty").alias("total_produk_terjual")
)

# Menjalankan streaming query
query_eksplorasi_3 = eksplorasi_3.writeStream \
    .format("memory") \
    .queryName("total_produk_cabang") \
    .outputMode("complete") \
    .start()

query_eksplorasi_3.processAllAvailable()

# Menampilkan hasil
spark.sql("""
    SELECT * FROM total_produk_cabang
    ORDER BY total_produk_terjual DESC
""").show()

+-----------+--------------------+
|nama_cabang|total_produk_terjual|
+-----------+--------------------+
|   Magelang|                   9|
| Yogyakarta|                   8|
|   Semarang|                   7|
|       Solo|                   7|
+-----------+--------------------+

